# LAION-fMRI benchmark — usage examples

How to score models on every flavor of the benchmark, from the headline 20 to the deep cuts.

**Conventions:**
- 20 *headline* benchmarks are registered in `benchmark_registry` and score via `brainscore_vision.score(model_id, benchmark_id)`.
- ~50 *non-headline* variants (cluster_k5, per-OOD-category, IT_full) live as factory functions; we call them directly.
- All paths produce a `Score` with the same structure: `score.values`, `score.attrs['raw_subjects']`, etc.

**Prereqs:**
```bash
conda activate vision-2026   # or your brain-score env
```
Stimuli must be available locally (auto-downloaded from S3 on first call) and AWS credentials must be configured for the gated stimulus bucket.

## 0. Inspect what's registered

In [1]:
from brainscore_vision import benchmark_registry
import brainscore_vision.benchmarks.laion_fmri  # populates registry

laion_ids = sorted([k for k in benchmark_registry if k.startswith('LAION_fMRI')])
print(f'Registered LAION-fMRI benchmarks: {len(laion_ids)}')
for k in laion_ids:
    print(f'  {k}')

/opt/anaconda3/envs/vision-2026/lib/python3.11/site-packages/brainscore_core/metrics/__init__.py:16: FutureWarning: xarray subclass Score should explicitly define __slots__
  class Score(DataAssembly):


Registered LAION-fMRI benchmarks: 20
  LAION_fMRI.IT-ood-ridge
  LAION_fMRI.IT-rdm-pearson
  LAION_fMRI.IT-tau-ridge
  LAION_fMRI.V1-ood-ridge
  LAION_fMRI.V1-rdm-pearson
  LAION_fMRI.V1-tau-ridge
  LAION_fMRI.V2-ood-ridge
  LAION_fMRI.V2-rdm-pearson
  LAION_fMRI.V2-tau-ridge
  LAION_fMRI.V4-ood-ridge
  LAION_fMRI.V4-rdm-pearson
  LAION_fMRI.V4-tau-ridge
  LAION_fMRI_persubject.IT-ood-ridge
  LAION_fMRI_persubject.IT-tau-ridge
  LAION_fMRI_persubject.V1-ood-ridge
  LAION_fMRI_persubject.V1-tau-ridge
  LAION_fMRI_persubject.V2-ood-ridge
  LAION_fMRI_persubject.V2-tau-ridge
  LAION_fMRI_persubject.V4-ood-ridge
  LAION_fMRI_persubject.V4-tau-ridge


## 1. Headline path — score via the registry

Standard Brain-Score interface. Works for any of the 20 headline IDs.

In [2]:
from brainscore_vision import _run_score

score = _run_score('alexnet', 'LAION_fMRI_persubject.IT-tau-ridge')
print(f"ceiled: {float(score.values):.4f}")
print(f"ceiling: {float(score.attrs['ceiling'].values):.4f}")
for sub in ['sub-01','sub-03','sub-05','sub-06','sub-07']:
    if sub in score.attrs:
        print(f"  {sub}: {float(score.attrs[sub].values):+.4f}")

/opt/anaconda3/envs/vision-2026/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/opt/anaconda3/envs/vision-2026/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
/opt/anaconda3/envs/vision-2026/lib/python3.11/site-packages/sklearn/linear_model/_ridge.py:253: LinAlgWarning: Ill-conditioned matrix (rcond=3.23719e-08): result may not be accurate.
  dual_coef = linalg.solve(K, y, assume_a="pos", overwrite_a=False)
/opt/anaconda3/envs/vision-2026/lib/python3.11/site-packages/sklearn/linear_model/_ri

ceiled: 0.0306
ceiling: 0.7483
  sub-01: +0.0242
  sub-03: +0.0404
  sub-05: +0.0244
  sub-06: +0.0296
  sub-07: +0.0346


## 2. Non-headline via factory — single-split variants

The `LAIONfMRI(region, split)` factory accepts any split name the upstream `laion_fmri.splits` package knows about — including per-OOD-category breakdowns the registry doesn't surface.

In [3]:
from brainscore_vision import load_model
from brainscore_vision.benchmarks.laion_fmri.benchmark import LAIONfMRI

model = load_model('alexnet')

# OOD: gabor patches only, IT region
bench = LAIONfMRI('IT', 'ood_gabor')
score = bench(model)
print(f'{bench.identifier}: {float(score.values):.4f}')

# OOD: illusion-natural only, V4
bench = LAIONfMRI('V4', 'ood_illusion-natural')
score = bench(model)
print(f'{bench.identifier}: {float(score.values):.4f}')

# IT_full ablation (V4 union IT) on tau split
bench = LAIONfMRI('IT_full', 'tau')
score = bench(model)
print(f'{bench.identifier}: {float(score.values):.4f}')

# Persubject pool variant
bench = LAIONfMRI('IT', 'ood', dataset_prefix='LAION_fMRI_persubject')
score = bench(model)
print(f'{bench.identifier}: {float(score.values):.4f}')

/opt/anaconda3/envs/vision-2026/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/opt/anaconda3/envs/vision-2026/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


LAION_fMRI.IT-ood_gabor-ridge: 0.0410


/opt/anaconda3/envs/vision-2026/lib/python3.11/site-packages/sklearn/linear_model/_ridge.py:253: LinAlgWarning: Ill-conditioned matrix (rcond=5.07851e-08): result may not be accurate.
  dual_coef = linalg.solve(K, y, assume_a="pos", overwrite_a=False)
/opt/anaconda3/envs/vision-2026/lib/python3.11/site-packages/sklearn/linear_model/_ridge.py:253: LinAlgWarning: Ill-conditioned matrix (rcond=5.07851e-08): result may not be accurate.
  dual_coef = linalg.solve(K, y, assume_a="pos", overwrite_a=False)
/opt/anaconda3/envs/vision-2026/lib/python3.11/site-packages/sklearn/linear_model/_ridge.py:253: LinAlgWarning: Ill-conditioned matrix (rcond=5.07851e-08): result may not be accurate.
  dual_coef = linalg.solve(K, y, assume_a="pos", overwrite_a=False)
/opt/anaconda3/envs/vision-2026/lib/python3.11/site-packages/sklearn/linear_model/_ridge.py:253: LinAlgWarning: Ill-conditioned matrix (rcond=5.07851e-08): result may not be accurate.
  dual_coef = linalg.solve(K, y, assume_a="pos", overwrite_a

LAION_fMRI.V4-ood_illusion-natural-ridge: 0.2275
LAION_fMRI.IT_full-tau-ridge: 0.1451


/opt/anaconda3/envs/vision-2026/lib/python3.11/site-packages/sklearn/linear_model/_ridge.py:253: LinAlgWarning: Ill-conditioned matrix (rcond=5.33326e-08): result may not be accurate.
  dual_coef = linalg.solve(K, y, assume_a="pos", overwrite_a=False)
/opt/anaconda3/envs/vision-2026/lib/python3.11/site-packages/sklearn/linear_model/_ridge.py:253: LinAlgWarning: Ill-conditioned matrix (rcond=2.85623e-08): result may not be accurate.
  dual_coef = linalg.solve(K, y, assume_a="pos", overwrite_a=False)
/opt/anaconda3/envs/vision-2026/lib/python3.11/site-packages/sklearn/linear_model/_ridge.py:253: LinAlgWarning: Ill-conditioned matrix (rcond=5.00713e-08): result may not be accurate.
  dual_coef = linalg.solve(K, y, assume_a="pos", overwrite_a=False)
/opt/anaconda3/envs/vision-2026/lib/python3.11/site-packages/sklearn/linear_model/_ridge.py:253: LinAlgWarning: Ill-conditioned matrix (rcond=3.84327e-08): result may not be accurate.
  dual_coef = linalg.solve(K, y, assume_a="pos", overwrite_a

LAION_fMRI_persubject.IT-ood-ridge: 0.0190


**Available split names** (defined by the upstream `laion_fmri.splits` package, re:vision framework):

| Split | Method | What it tests |
|---|---|---|
| `tau` | re:vision Method 1 | Within-distribution, MMD-matched random split |
| `ood` | re:vision Method 3 | All 371 curated OOD images (9 categories pooled) |
| `ood_shape` / `ood_relations` / `ood_unusual` / `ood_cropped` / `ood_selfmade` / `ood_gaudy` / `ood_illusion-classic` / `ood_gabor` / `ood_illusion-natural` | Method 3 sub-categories | Single OOD-category test |

**Regions**: `V1`, `V2`, `V4`, `IT`, and the runtime alias `IT_full` (= V4 ∪ IT).

## 3. Cluster-CV variants (re:vision Method 2)

5-fold CV on CLIP-cluster hold-out. Heavier than tau/ood (5 fits per call). Use `LAIONfMRIClusterCV`.

In [4]:
from brainscore_vision.benchmarks.laion_fmri.benchmark import LAIONfMRIClusterCV

# Shared pool, V4, all 5 subjects
bench = LAIONfMRIClusterCV('V4')
score = bench(model)
print(f'{bench.identifier}: {float(score.values):.4f}')
print(f"per-fold scores: {[float(s) for s in score.attrs['raw_folds'].values]}")

# Persubject pool — heaviest variant, takes ~30+ min per region
bench = LAIONfMRIClusterCV('IT', dataset_prefix='LAION_fMRI_persubject')
score = bench(model)

/opt/anaconda3/envs/vision-2026/lib/python3.11/site-packages/sklearn/linear_model/_ridge.py:253: LinAlgWarning: Ill-conditioned matrix (rcond=5.95406e-08): result may not be accurate.
  dual_coef = linalg.solve(K, y, assume_a="pos", overwrite_a=False)
/opt/anaconda3/envs/vision-2026/lib/python3.11/site-packages/sklearn/linear_model/_ridge.py:253: LinAlgWarning: Ill-conditioned matrix (rcond=5.95406e-08): result may not be accurate.
  dual_coef = linalg.solve(K, y, assume_a="pos", overwrite_a=False)
/opt/anaconda3/envs/vision-2026/lib/python3.11/site-packages/sklearn/linear_model/_ridge.py:253: LinAlgWarning: Ill-conditioned matrix (rcond=5.95406e-08): result may not be accurate.
  dual_coef = linalg.solve(K, y, assume_a="pos", overwrite_a=False)
/opt/anaconda3/envs/vision-2026/lib/python3.11/site-packages/sklearn/linear_model/_ridge.py:253: LinAlgWarning: Ill-conditioned matrix (rcond=5.95406e-08): result may not be accurate.
  dual_coef = linalg.solve(K, y, assume_a="pos", overwrite_a

LAION_fMRI.V4-cluster_k5-ridge: 0.2389
per-fold scores: [0.29843910367219106, 0.23126798326222447, 0.30684717955407337, 0.15351057467280188, 0.20440844654499016]


/opt/anaconda3/envs/vision-2026/lib/python3.11/site-packages/sklearn/linear_model/_ridge.py:253: LinAlgWarning: Ill-conditioned matrix (rcond=5.77483e-08): result may not be accurate.
  dual_coef = linalg.solve(K, y, assume_a="pos", overwrite_a=False)
/opt/anaconda3/envs/vision-2026/lib/python3.11/site-packages/sklearn/linear_model/_ridge.py:253: LinAlgWarning: Ill-conditioned matrix (rcond=4.26691e-08): result may not be accurate.
  dual_coef = linalg.solve(K, y, assume_a="pos", overwrite_a=False)
/opt/anaconda3/envs/vision-2026/lib/python3.11/site-packages/sklearn/linear_model/_ridge.py:253: LinAlgWarning: Ill-conditioned matrix (rcond=5.15662e-08): result may not be accurate.
  dual_coef = linalg.solve(K, y, assume_a="pos", overwrite_a=False)
/opt/anaconda3/envs/vision-2026/lib/python3.11/site-packages/sklearn/linear_model/_ridge.py:253: LinAlgWarning: Ill-conditioned matrix (rcond=5.384e-08): result may not be accurate.
  dual_coef = linalg.solve(K, y, assume_a="pos", overwrite_a=F

## 4. Single-subject scoring (any variant)

All factories accept a `subjects` tuple. Pass one subject for fast diagnostics or per-individual analysis.

In [5]:
# Score sub-03 alone on persubject V4-tau
bench = LAIONfMRI('V4', 'tau', dataset_prefix='LAION_fMRI_persubject', subjects=('sub-03',))
score = bench(model)
print(f'{bench.identifier} (sub-03 only): {float(score.values):.4f}')

# Subset of subjects
bench = LAIONfMRI('IT', 'tau', subjects=('sub-01','sub-05','sub-07'))
score = bench(model)

/opt/anaconda3/envs/vision-2026/lib/python3.11/site-packages/sklearn/linear_model/_ridge.py:255: UserWarning: Singular matrix in solving dual problem. Using least-squares solution instead.
  warnings.warn(


LAION_fMRI_persubject.V4-tau-ridge (sub-03 only): 0.1699


## 5. RSA variants (representational alignment)

RSA compares the model's RDM to each subject's neural RDM (Spearman r). Only registered for the shared pool — persubject RSA isn't supported because Nili ceiling requires shared stimuli across subjects.

In [6]:
from brainscore_vision.benchmarks.laion_fmri.benchmark import LAIONfMRIRSA

# Shared pool RSA (registered headline)
bench = LAIONfMRIRSA('IT')
score = bench(model)
print(f'{bench.identifier}: ceiled {float(score.values):.4f}  raw {float(score.attrs["raw"].values):.4f}')

# Will raise ValueError on persubject:
try:
    LAIONfMRIRSA('IT', dataset_prefix='LAION_fMRI_persubject')
except ValueError as e:
    print(f'Expected error: {e}')

/opt/anaconda3/envs/vision-2026/lib/python3.11/site-packages/xarray/core/concat.py:500: FutureWarning: unique with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.
  common_dims = tuple(pd.unique([d for v in vars for d in v.dims]))
/opt/anaconda3/envs/vision-2026/lib/python3.11/site-packages/xarray/core/concat.py:500: FutureWarning: unique with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.
  common_dims = tuple(pd.unique([d for v in vars for d in v.dims]))
/opt/anaconda3/envs/vision-2026/lib/python3.11/site-packages/xarray/core/concat.py:500: FutureWarning: unique with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.
  common_dims = tuple(pd.unique([d for v in vars for d in v.dims]))
/opt/anaconda3/envs/vision-2026/lib/python3.11/site-packages/xarray/core/concat.py:50

LAION_fMRI.IT-rdm-pearson: ceiled 0.3745  raw 0.2815
Expected error: LAIONfMRIRSA is only defined for the shared pool. Persubject stimuli differ across subjects → no Nili ceiling.


/opt/anaconda3/envs/vision-2026/lib/python3.11/site-packages/xarray/core/concat.py:500: FutureWarning: unique with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.
  common_dims = tuple(pd.unique([d for v in vars for d in v.dims]))


## 6. Tuning the noise-ceiling threshold

Voxels are filtered by `nc_12rep` (% R² from the published ncsnr-based reliability — values stored as percentages, 0-100). Default threshold is **30.0** (keep only voxels with ≥ 30% explainable variance). Lower for broader voxel inclusion, higher for stricter.

In [7]:
# Looser threshold (15% R²) — more voxels included, individually less reliable
bench = LAIONfMRI('V4', 'tau', noise_ceiling_threshold=15.0)
score = bench(model)
print(f'NC=15: {float(score.values):.4f}  ceiling {float(score.attrs["ceiling"].values):.4f}')

# Stricter (50% R²) — fewer voxels, each highly reliable
bench = LAIONfMRI('V4', 'tau', noise_ceiling_threshold=50.0)
score = bench(model)
print(f'NC=50: {float(score.values):.4f}  ceiling {float(score.attrs["ceiling"].values):.4f}')

activations:   0%|          | 0/960 [00:00<?, ?it/s]

layer packaging:   0%|          | 0/1 [00:00<?, ?it/s]

activations:   0%|          | 0/256 [00:00<?, ?it/s]

layer packaging:   0%|          | 0/1 [00:00<?, ?it/s]

activations:   0%|          | 0/960 [00:00<?, ?it/s]

...

activations:   0%|          | 0/960 [00:00<?, ?it/s]

layer packaging:   0%|          | 0/1 [00:00<?, ?it/s]

activations:   0%|          | 0/256 [00:00<?, ?it/s]

layer packaging:   0%|          | 0/1 [00:00<?, ?it/s]

NC=15: 0.2534  ceiling 0.7993
NC=50: 0.3358  ceiling 0.7993


## 7. Bulk-register non-headline variants for your session

If you want non-headline variants to behave like first-class registry entries (e.g., for batch scoring in your notebook), add them to `benchmark_registry` at import time:

> **Caveat:** Brain-Score's `_run_score` calls `import_plugin` which does a text-grep through plugin source files for `benchmark_registry['<id>']` literals. Runtime-added entries aren't text-discoverable, so `_run_score('alexnet', 'LAION_fMRI.IT-ood_gabor-ridge')` will fail with `No registrations found`. Invoke them via the registry dict directly: `benchmark_registry[<id>]()`. The factories return regular benchmark objects you can call with `benchmark(model)`.

In [10]:
from brainscore_vision import benchmark_registry

OOD_TYPES = ['shape', 'relations', 'unusual', 'cropped', 'selfmade',
             'gaudy', 'illusion-classic', 'gabor', 'illusion-natural']

# All cluster_k5 variants (8 — 4 regions x 2 families)
for region in ['V1', 'V2', 'V4', 'IT']:
    for fam in ['LAION_fMRI', 'LAION_fMRI_persubject']:
        bid = f'{fam}.{region}-cluster_k5-ridge'
        benchmark_registry[bid] = (
            lambda r=region, f=fam: LAIONfMRIClusterCV(r, dataset_prefix=f)
        )

# Per-OOD-category breakdowns (36 — 4 regions x 9 categories, shared only)
for region in ['V1', 'V2', 'V4', 'IT']:
    for t in OOD_TYPES:
        bid = f'LAION_fMRI.{region}-ood_{t}-ridge'
        benchmark_registry[bid] = (lambda r=region, t=t: LAIONfMRI(r, f'ood_{t}'))

# IT_full (V4 union IT) for all splits
for split in ['tau', 'ood'] + [f'ood_{t}' for t in OOD_TYPES]:
    bid = f'LAION_fMRI.IT_full-{split}-ridge'
    benchmark_registry[bid] = (lambda s=split: LAIONfMRI('IT_full', s))

print(f'Total LAION benchmarks now registered: '
      f'{len([k for k in benchmark_registry if k.startswith("LAION_fMRI")])}')

Total LAION benchmarks now registered: 75


In [11]:
# Brain-Score's `_run_score` / `load_benchmark` does a text-grep through plugin
# source files for `benchmark_registry['<id>']` literals to find the file owning
# the benchmark — runtime-registered entries aren't text-discoverable. So for
# runtime additions we skip that path and invoke from the registry dict directly:
benchmark = benchmark_registry['LAION_fMRI.IT-ood_gabor-ridge']()
score = benchmark(model)
print(f'IT ood_gabor: {float(score.values):.4f}')

IT ood_gabor: 0.0410


## 8. Pre-computed results

The 5-model published baseline lives next to the sweep runner in [`baselines/`](baselines/):
- `baseline_sweep_results.csv` — long form, 113 rows (model, benchmark, score, ceiling, time, error)
- `baseline_sweep_pivot.csv` — wide pivot, 20 lean headlines × 5 models

Models scored: alexnet, alexnet_random, resnet50_tutorial, convnext_tiny_imagenet1K_torchvisionV1, resnext101_32x8d_wsl.

In [ ]:
import pandas as pd
from pathlib import Path

# CSVs ship inside the plugin so this works from anywhere the plugin is
# importable. Resolve via the plugin module's path.
import brainscore_vision.benchmarks.laion_fmri as _bench_pkg
results_dir = Path(_bench_pkg.__file__).parent / "baselines"

long_df = pd.read_csv(results_dir / "baseline_sweep_results.csv")
pivot_df = pd.read_csv(results_dir / "baseline_sweep_pivot.csv", index_col=[0, 1, 2])

print(f"Long form: {len(long_df)} rows")
print(f"Pivot: {pivot_df.shape}")
pivot_df.round(3)

## 9. Gotchas

- **First call per (model, stim_set) is slow**: activations get extracted and cached at `~/.result_caching/`. Subsequent calls hit the cache (~30s vs ~10min).
- **Persubject cells download big**: each subject's `LAION_fMRI_persubject_full_sub-XX` assembly is ~400-800 MB. Cached at `~/.brainio/` after first fetch.
- **Stimuli are DUA-gated**: AWS credentials needed for the private S3 prefix. Brain-Score will auto-download on first call if creds are configured.
- **MacOS jetsam**: the heaviest cells (cluster_k5 persubject IT for big models) can hit ~30GB RAM during ridge fitting. macOS may SIGKILL them. Either use a single-process Python invocation per cell (see `baselines/sweep_per_cell.sh`) or run on a Linux box with more RAM headroom.